# Basics + Contracts

Paxman supports four contract formats. This notebook demonstrates the same normalization using all four, proving the output is identical regardless of format.

## 1. Pydantic

In [ ]:
from pydantic import BaseModel

import paxman
import paxman.contract.adapters.dict_dsl
import paxman.contract.adapters.json_schema
import paxman.contract.adapters.openapi
import paxman.contract.adapters.pydantic


class Supplier(BaseModel):
    name: str
    country: str


SAMPLE = "ACME Corp, based in Malaysia"
result_pydantic = paxman.normalize(SAMPLE, Supplier)
print("Pydantic:", result_pydantic.normalized_data)

## 2. JSON Schema

In [ ]:
json_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "country": {"type": "string"},
    },
    "required": ["name", "country"],
}

result_json = paxman.normalize(SAMPLE, json_schema)
print("JSON Schema:", result_json.normalized_data)

## 3. Dict DSL

In [ ]:
dict_dsl = {
    "fields": {
        "name": {"type": "string", "description": "Supplier name"},
        "country": {"type": "string", "description": "Country of operation"},
    }
}

result_dsl = paxman.normalize(SAMPLE, dict_dsl)
print("Dict DSL:", result_dsl.normalized_data)

## 4. OpenAPI

In [ ]:
openapi_spec = {
    "openapi": "3.0.0",
    "info": {"title": "Supplier", "version": "1.0.0"},
    "paths": {},
    "components": {
        "schemas": {
            "Supplier": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "country": {"type": "string"},
                },
                "required": ["name", "country"],
            }
        }
    },
}

result_oas = paxman.normalize(SAMPLE, openapi_spec)
print("OpenAPI:", result_oas.normalized_data)

In [ ]:
# Verify all four produce identical output
expected = result_pydantic.normalized_data
assert result_json.normalized_data == expected, "JSON Schema mismatch"
assert result_dsl.normalized_data == expected, "Dict DSL mismatch"
assert result_oas.normalized_data == expected, "OpenAPI mismatch"
print("All four contract formats produce identical normalized data.")

## Key takeaways

- The **contract adapter layer** translates external schemas into Paxman's internal `CanonicalContract`.
- All adapters are **optional extras** — import them to register, or Paxman only ships the Dict DSL.
- The output is **format-independent** — same contract semantics, same result.

> **Reference:** See `docs/concepts/contracts.md` for the full contract model.

## Try it yourself

- Add a new field (e.g. `registration_number`) to all four contracts and see each adapt.
- Try an OpenAPI 3.1 spec instead of 3.0.
- What happens if you omit a required field from the JSON Schema?